# 🏠 House Price Analysis & Prediction
## Main Report — End-to-End ML Pipeline

---

| Item | Detail |
|------|--------|
| **Dataset** | KC House Data (kc_house_data.csv) |
| **Target** | `price` (continuous — Regression) |
| **Models** | Linear Regression · Random Forest · Neural Network |
| **Version** | 2.0 · April 2026 |

---

### Pipeline Overview
```
Data Ingestion → Preprocessing → EDA → Feature Eng. → Train → Evaluate → Deploy
```

In [ ]:
# ── CELL 1 : IMPORTS ─────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection  import train_test_split, cross_val_score
from sklearn.preprocessing    import StandardScaler
from sklearn.pipeline         import Pipeline
from sklearn.impute           import SimpleImputer
from sklearn.linear_model     import LinearRegression
from sklearn.ensemble         import RandomForestRegressor
from sklearn.neural_network   import MLPRegressor
from sklearn.metrics          import r2_score, mean_absolute_error, mean_squared_error

# Compatibility fix
if tuple(map(int, sklearn.__version__.split('.')[:2])) >= (1, 4):
    from sklearn.metrics import root_mean_squared_error
else:
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Shared constants (consistent across all files) ───────────
FEATURES = ['sqft_living','grade','bathrooms','view','sqft_basement','lat','bedrooms']
TARGET   = 'price'

print(f'scikit-learn version : {sklearn.__version__}')
print(f'Features selected    : {FEATURES}')

## 📥 Section 1 — Data Ingestion

In [ ]:
# ── CELL 2 : LOAD DATA ───────────────────────────────────────
# Place kc_house_data.csv in the same folder as this notebook
CSV_PATH = 'kc_house_data.csv'
df = pd.read_csv(CSV_PATH)

print(f'Shape   : {df.shape}')
print(f'Columns : {list(df.columns)}')
df.head()

In [ ]:
# ── CELL 3 : BASIC INFO ──────────────────────────────────────
print('── Data Types ──')
print(df.dtypes)
print('\n── Missing Values ──')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n── Descriptive Statistics ──')
df[FEATURES + [TARGET]].describe().round(2)

## 🔧 Section 2 — Preprocessing

In [ ]:
# ── CELL 4 : PREPROCESSING ───────────────────────────────────
# Drop duplicates
before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed : {before - len(df)}')

# Select features + target
df_model = df[FEATURES + [TARGET]].copy()

# Impute missing values
imputer = SimpleImputer(strategy='mean')
df_model[FEATURES] = imputer.fit_transform(df_model[FEATURES])
print(f'Nulls after impute : {df_model.isnull().sum().sum()}')

# Outlier clipping (IQR)
for col in FEATURES:
    Q1, Q3 = df_model[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_model[col] = df_model[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

print(f'Feature matrix     : {df_model[FEATURES].shape}')
print('Preprocessing complete ✔')

## 📊 Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── CELL 5 : PRICE DISTRIBUTION ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_model[TARGET]/1e6, bins=60, color='#00e5ff', edgecolor='#0d1220', alpha=0.85)
axes[0].set_xlabel('Price (Millions $)')
axes[0].set_ylabel('Count')
axes[0].set_title('House Price Distribution')

axes[1].hist(np.log1p(df_model[TARGET]), bins=60, color='#a78bfa', edgecolor='#0d1220', alpha=0.85)
axes[1].set_xlabel('log(Price)')
axes[1].set_title('log(Price) Distribution — more normal')

for ax in axes:
    ax.set_facecolor('#111827')
fig.patch.set_facecolor('#05080f')
plt.suptitle('Price Distribution', fontsize=13, color='#e2e8f0')
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 6 : CORRELATION HEATMAP ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
corr = df_model[FEATURES + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13)
fig.patch.set_facecolor('#05080f')
ax.set_facecolor('#111827')
plt.tight_layout()
plt.show()

# Top correlated features with price
print('\nTop correlations with price:')
print(corr[TARGET].sort_values(ascending=False).to_string())

In [ ]:
# ── CELL 7 : FEATURE vs PRICE SCATTER ────────────────────────
top_features = ['sqft_living', 'grade', 'bathrooms', 'lat']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

colors = ['#00e5ff', '#a78bfa', '#34d399', '#f59e0b']
for ax, col, color in zip(axes.flatten(), top_features, colors):
    ax.scatter(df_model[col], df_model[TARGET]/1e6,
               alpha=0.25, s=6, color=color)
    ax.set_xlabel(col); ax.set_ylabel('Price (M$)')
    ax.set_title(f'{col} vs Price')
    ax.set_facecolor('#111827')

fig.patch.set_facecolor('#05080f')
plt.suptitle('Feature vs Price Relationships', fontsize=13, color='#e2e8f0')
plt.tight_layout()
plt.show()

## ⚙️ Section 4 — Model Training (LR · RF · NN)

In [ ]:
# ── CELL 8 : TRAIN / TEST SPLIT ──────────────────────────────
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train set : {X_train.shape}')
print(f'Test set  : {X_test.shape}')

In [ ]:
# ── CELL 9 : BUILD MODELS (all wrapped in Pipeline + Scaler) ──
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  RandomForestRegressor(
            n_estimators=200, max_depth=None,
            min_samples_split=5, random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    'Neural Network': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu',
            max_iter=1000,
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20
        ))
    ]),
}
print(f'Models defined : {list(models.keys())}')

In [ ]:
# ── CELL 10 : TRAIN + CROSS-VALIDATE ─────────────────────────
trained = {}
for name, pipeline in models.items():
    print(f'Training {name} …')
    pipeline.fit(X_train, y_train)
    cv = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    trained[name] = {'pipeline': pipeline, 'cv_mean': cv.mean(), 'cv_std': cv.std()}
    print(f'  CV R² = {cv.mean():.4f} ± {cv.std():.4f}')

print('\nAll models trained ✔')

## 📈 Section 5 — Evaluation & Model Comparison

In [ ]:
# ── CELL 11 : EVALUATION TABLE ───────────────────────────────
rows = []
for name, res in trained.items():
    preds  = res['pipeline'].predict(X_test)
    r2     = r2_score(y_test, preds)
    rmse   = root_mean_squared_error(y_test, preds)
    mae    = mean_absolute_error(y_test, preds)
    status = '✓ PASS' if r2 >= 0.80 else '⚠ TUNE'
    rows.append({
        'Model':  name,
        'R²':     round(r2,   4),
        'RMSE':   round(rmse, 0),
        'MAE':    round(mae,  0),
        'CV R²':  round(res['cv_mean'], 4),
        'Status': status
    })

results_df = pd.DataFrame(rows).sort_values('R²', ascending=False).reset_index(drop=True)
print('── Model Comparison Table ──')
results_df

In [ ]:
# ── CELL 12 : COMPARISON BAR CHART ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = ['R²', 'RMSE', 'MAE']
colors  = ['#34d399', '#00e5ff', '#a78bfa']

for ax, metric in zip(axes, metrics):
    bars = ax.bar(results_df['Model'], results_df[metric],
                  color=colors, edgecolor='#1e2d45', width=0.5)
    ax.set_title(f'Model {metric}')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    ax.set_facecolor('#111827')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.01,
                f'{bar.get_height():,.0f}',
                ha='center', va='bottom', fontsize=8, color='#e2e8f0')

fig.patch.set_facecolor('#05080f')
plt.suptitle('Model Performance Comparison', fontsize=13, color='#e2e8f0')
plt.tight_layout()
plt.show()

## 🚀 Section 6 — Save Best Model

In [ ]:
# ── CELL 13 : SAVE MODELS ────────────────────────────────────
os.makedirs('ml_output/models', exist_ok=True)

try:
    for name, res in trained.items():
        fname = name.lower().replace(' ', '_') + '.pkl'
        joblib.dump(res['pipeline'], f'ml_output/models/{fname}')
        print(f'Saved : {fname}')
    results_df.to_csv('ml_output/results_summary.csv', index=False)
    print('Saved : results_summary.csv')
    print('\nAll models saved ✔')
except Exception as e:
    print(f'Save error: {e}')

## 📋 Summary

| Model | Strength | Weakness |
|-------|----------|----------|
| **Linear Regression** | Fast, interpretable, good baseline | Assumes linearity, may underfit |
| **Random Forest** | Handles non-linearity, robust | Slower, less interpretable |
| **Neural Network** | Captures complex patterns | Needs scaling, more epochs to tune |

### Next Steps
- Hyperparameter tuning with `GridSearchCV` for RF and NN
- Add features: `zipcode`, `yr_renovated`, `waterfront`
- 5-Fold Cross-Validation comparison across all models
- SHAP feature importance analysis
- Deploy best model as a REST API (Flask / FastAPI)

> See **Technical Appendix** notebook for deep-dive model analysis.